 
# First-Order Leakage Assessment (TVLA)

This notebook implements a **fixed-vs-random Test Vector Leakage Assessment (TVLA)** to evaluate first-order statistical leakage in side-channel measurements.

The analysis processes labeled HDF5 traces using memory-efficient batch accumulation and computes Welch's *t*-statistic independently at each sample point. Potential leakage is assessed against the conventional **$\lvert t\rvert > 4.5$** threshold.

To support large-scale experiments, the implementation includes input validation, persistent checkpoints, and automatic recovery from interrupted executions. Results are summarized numerically and visualized as a sample-wise *t*-statistic trace.

**Usage:** Configure the dataset paths and acquisition parameters in the configuration cell before running the analysis.

## Input contract and experimental configuration

For each entry of `FOLDERS_LIST`, the notebook discovers `.h5` files from the following directory structure (nested experiment folders are supported):

```text
</path/to/dataset_root>/
└── <trace_collection>/
    └── <experiment_group>/
        └── <experiment_run>/
            ├── <capture_001>.h5
            ├── <capture_002>.h5
            └── ...
```

The three corresponding placeholders are `base_path`, `FOLDERS_LIST`, and `EXPERIMENT_NAME`. Replace all three in **cell 1** with paths for the local dataset. The relative hierarchy and names must match the actual files; adapt path separators for the operating system. The values shown in the first cell are deliberately non-identifying and **are not valid dataset locations**.

Each HDF5 file must contain:

| Dataset | Required shape | Interpretation |
| --- | --- | --- |
| `traces` | `(N, trace_length)` | Recorded sample values; all columns are analyzed. |
| `labels` | `(N,)` | Group membership: **0 = fixed** and **1 = random**. |

The unchanged implementation requires equal numbers of trace rows and labels, accepts only group labels 0 and 1, and skips invalid files subject to its existing resume-state safeguards. It reads the traces in file-name order, then dataset order, and casts each batch to `float64` before updating its accumulators. It applies **no filtering, cropping, normalization, realignment, or resampling**.


In [ ]:
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import pickle
import os
from tqdm import tqdm
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa
from tqdm.notebook import trange
import pickle

# ============================================================================
# CONFIGURATION — EDIT THIS CELL BEFORE RUNNING THE NOTEBOOK
# ============================================================================
# REQUIRED (1/3): Replace this anonymized placeholder with the absolute path
# to the dataset root. The same directory is also used for saved TVLA states,
# checkpoints, and logs. Use the appropriate path syntax for your operating
# system (e.g., a raw Windows path or a POSIX path).
base_path = r"/path/to/dataset_root"
# REQUIRED (2/3): Replace with the actual first-level trace directory names
# located under base_path. Keep a list even if there is only one directory.
FOLDERS_LIST = ["trace_collection"]
# REQUIRED (3/3): Replace with the relative experiment directory below each
# trace collection. Include any nested subdirectories; use your OS path
# separator (e.g., "group/run" on POSIX or r"group\run" on Windows).
# Do not include base_path or a filename in EXPERIMENT_NAME.
EXPERIMENT_NAME = r"experiment_group/experiment_run"
# DATA COMPATIBILITY CHECK: The number of samples in each HDF5 trace must
# equal trace_length. The original value is preserved for reproducibility.
trace_length = 40_000
# ============================================================================
# All remaining values are preserved from the original analysis.
# Review the limits and state flags only if deliberately configuring a new
# experimental run; changing them may change which traces are included.
# ============================================================================

complete_traces_list = []

traces_per_folder = 1000
num_traces_per_folder = 1000

MAX_TRACES = 20_000_000
MAX_FILES = 3000

SAVE_EVERY_TRACES = 1_000

# Decision threshold: |t| > 4.5 (unchanged).
TVLA_THRESHOLD = 4.5

# Memory budget for processing batches (MB)
BATCH_MEMORY_MB = 256

# State management: True deletes the derived state/checkpoints for this run.
# Restart from scratch if True
RESET_STATE = False

# Reuse full-length original state if available
REUSE_ORIGINAL_STATE = True


### First-Order Fixed-vs-Random TVLA

This cell performs a **sample-wise, first-order Test Vector Leakage Assessment (TVLA)** using Welch's two-sample *t*-test on HDF5 traces partitioned into fixed (`label = 0`) and random (`label = 1`) groups. At each sample index $j$, the statistic is

$$
t_j = \frac{\bar{x}_{f,j}-\bar{x}_{r,j}}{\sqrt{\frac{s_{f,j}^{2}}{n_f}+\frac{s_{r,j}^{2}}{n_r}}}.
$$

Here, $\bar{x}$, $s^2$, and $n$ denote the sample mean, unbiased sample variance, and group size, respectively. Per-sample sums and squared sums are accumulated in `float64` over memory-bounded batches, without retaining the complete trace set. HDF5 structure, trace dimensions, and class labels are validated before inclusion.

**Checkpointing and recovery.** At eligible `SAVE_EVERY_TRACES` milestones (checked after successfully processing a file), the accumulators, group counts, completed-file list, current-file offset, and $t$-values are saved as `.npz` checkpoints under `base_path/tvla_checkpoints_all_<safe_name>/tvla_all_<N>_traces.npz`. The latest state is maintained at `base_path/tvla_state_all_<safe_name>.npz` and is also saved on exit. Writes use a temporary file followed by atomic replacement. Unless `RESET_STATE` is enabled, rerunning the cell loads the latest state, skips completed files, and resumes any partially processed file from its saved offset; invalid-file contributions are rolled back.

The resulting $t_j$ sequence is plotted against the **$\pm 4.5$** thresholds, and the maximum $|t_j|$ and number of exceedances are reported. An exceedance indicates statistically detectable fixed-versus-random differences under this test.


In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


# ============================================================
# STATE FILES
# ============================================================

ORIGINAL_STATE_FILE = os.path.join(
    base_path,
    f"tvla_state_{EXPERIMENT_NAME}.npz"
)

safe_name = (
    EXPERIMENT_NAME
    .replace("\\", "_")
    .replace("/", "_")
)

STATE_FILE = os.path.join(
    base_path,
    f"tvla_state_all_{safe_name}.npz"
)

CHECKPOINT_DIR = os.path.join(
    base_path,
    f"tvla_checkpoints_all_{safe_name}"
)

# Log of invalid HDF5 files
SKIPPED_LOG_FILE = os.path.join(
    base_path,
    f"tvla_skipped_h5_{safe_name}.txt"
)

skipped_invalid_files = []


def log_invalid_file(h5_path, reason):
    """
    Record an invalid HDF5 file without marking it processed.
    """
    message = f"{h5_path} | {reason}"

    skipped_invalid_files.append(message)

    tqdm.write(
        f"\n[SKIP INVALID H5] {os.path.basename(h5_path)}"
    )
    tqdm.write(f"Reason: {reason}")

    try:
        os.makedirs(base_path, exist_ok=True)

        with open(
            SKIPPED_LOG_FILE,
            "a",
            encoding="utf-8"
        ) as log:
            log.write(message + "\n")

    except OSError as error:
        tqdm.write(
            f"[WARNING] Could not write skip log: {error}"
        )


# Automatic batch size based on trace length
BATCH_SIZE = max(
    1,
    int(
        BATCH_MEMORY_MB * 1024**2
        // (3 * trace_length * 8)
    )
)

print(f"Samples per trace: {trace_length:,}")
print(f"Traces per batch:  {BATCH_SIZE:,}")


# ============================================================
# CALCULATE TVLA
# ============================================================

def calculate_t_values():

    if count_f < 2 or count_r < 2:
        return np.full(trace_length, np.nan)

    mean_f = sum_f / count_f
    mean_r = sum_r / count_r

    var_f = np.maximum(
        (sum2_f - sum_f * mean_f) / (count_f - 1),
        0.0
    )

    var_r = np.maximum(
        (sum2_r - sum_r * mean_r) / (count_r - 1),
        0.0
    )

    denominator = np.sqrt(
        var_f / count_f +
        var_r / count_r
    )

    t_values = np.divide(
        mean_f - mean_r,
        denominator,
        out=np.full(trace_length, np.nan),
        where=denominator > 0
    )

    return t_values


# ============================================================
# SAVE STATE ATOMICALLY
# ============================================================

def save_state(path):

    os.makedirs(
        os.path.dirname(path),
        exist_ok=True
    )

    temporary = path + ".tmp.npz"

    np.savez(
        temporary,

        trace_length=np.int64(trace_length),

        sum_f=sum_f,
        sum_r=sum_r,

        sum2_f=sum2_f,
        sum2_r=sum2_r,

        count_f=np.int64(count_f),
        count_r=np.int64(count_r),
        count_total=np.int64(count_total),

        processed_files=np.asarray(
            sorted(processed_files),
            dtype=np.str_
        ),

        active_file=np.asarray(active_file),
        active_offset=np.int64(active_offset),

        t_values=calculate_t_values()
    )

    os.replace(temporary, path)


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_checkpoint_if_needed():

    if (
        count_total > 0
        and count_total % SAVE_EVERY_TRACES == 0
    ):

        checkpoint = os.path.join(
            CHECKPOINT_DIR,
            f"tvla_all_{count_total:09d}_traces.npz"
        )

        save_state(checkpoint)
        save_state(STATE_FILE)

        tqdm.write(
            f"\n[CHECKPOINT] "
            f"{count_total:,} traces saved -> {checkpoint}"
        )


# ============================================================
# LOAD PREVIOUS STATE
# ============================================================

if RESET_STATE:

    if os.path.exists(STATE_FILE):
        os.remove(STATE_FILE)

    if os.path.isdir(CHECKPOINT_DIR):

        for filename in os.listdir(CHECKPOINT_DIR):

            if (
                filename.startswith("tvla_all_")
                and filename.endswith("_traces.npz")
            ):

                os.remove(
                    os.path.join(
                        CHECKPOINT_DIR,
                        filename
                    )
                )


if not RESET_STATE and os.path.exists(STATE_FILE):

    source_file = STATE_FILE

elif (
    not RESET_STATE
    and REUSE_ORIGINAL_STATE
    and os.path.exists(ORIGINAL_STATE_FILE)
):

    source_file = ORIGINAL_STATE_FILE

else:

    source_file = None


if source_file is not None:

    with np.load(source_file, allow_pickle=False) as state:

        saved_length = int(state["trace_length"])

        if saved_length != trace_length:

            raise RuntimeError(
                f"Incompatible trace length. "
                f"Saved: {saved_length:,}, "
                f"Current: {trace_length:,}"
            )

        sum_f = state["sum_f"].copy()
        sum_r = state["sum_r"].copy()

        sum2_f = state["sum2_f"].copy()
        sum2_r = state["sum2_r"].copy()

        count_f = int(state["count_f"])
        count_r = int(state["count_r"])
        count_total = int(state["count_total"])

        processed_files = set(
            state["processed_files"].astype(str).tolist()
        )

        active_file = (
            str(state["active_file"])
            if "active_file" in state.files
            else ""
        )

        active_offset = (
            int(state["active_offset"])
            if "active_offset" in state.files
            else 0
        )

    print("\nPrevious TVLA state loaded:")
    print(os.path.abspath(source_file))

else:

    sum_f = np.zeros(trace_length, dtype=np.float64)
    sum_r = np.zeros(trace_length, dtype=np.float64)

    sum2_f = np.zeros(trace_length, dtype=np.float64)
    sum2_r = np.zeros(trace_length, dtype=np.float64)

    count_f = 0
    count_r = 0
    count_total = 0

    processed_files = set()

    active_file = ""
    active_offset = 0

    print("\nStarting TVLA from zero.")


if active_file and (
    active_offset <= 0
    or active_file in processed_files
):

    raise RuntimeError(
        "Inconsistent resume state."
    )


# ============================================================
# INITIAL INFORMATION
# ============================================================

initial_count_total = count_total
initial_files = len(processed_files)

print()
print("=" * 60)
print("INITIAL STATE")
print("=" * 60)

print(f"Samples per trace:     {trace_length:,}")
print(f"Processed files:       {len(processed_files):,}")
print(f"Processed traces:      {count_total:,}")

print(f"MAX_FILES:             {MAX_FILES:,}")
print(f"MAX_TRACES:            {MAX_TRACES:,}")
print(f"SAVE_EVERY_TRACES:     {SAVE_EVERY_TRACES:,}")

print(f"Active file:           {active_file or 'None'}")
print(f"Active offset:         {active_offset:,}")

print("=" * 60)


# ============================================================
# FIND FILES AND CHECK PREVIOUS PROCESSING
# ============================================================

all_pending = []

for folder_name in FOLDERS_LIST:

    folder_path = os.path.join(
        base_path,
        folder_name,
        EXPERIMENT_NAME
    )

    h5_files = sorted(
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".h5")
    )

    current_keys = {
        os.path.join(folder_name, f)
        for f in h5_files
    }

    saved_folder_keys = {
        key for key in processed_files
        if os.path.dirname(key) == folder_name
    }

    matching = current_keys & saved_folder_keys
    pending = current_keys - saved_folder_keys
    missing = saved_folder_keys - current_keys

    print()
    print("=" * 60)
    print(f"FILE CHECK [{folder_name}]")
    print("=" * 60)

    print(f"Files in directory:           {len(h5_files):,}")
    print(f"Files in saved state:         {len(saved_folder_keys):,}")
    print(f"Matching processed files:     {len(matching):,}")
    print(f"Pending files:                {len(pending):,}")
    print(f"Saved files not in directory: {len(missing):,}")

    print("=" * 60)

    for filename in h5_files:

        file_key = os.path.join(
            folder_name,
            filename
        )

        if file_key not in processed_files:

            h5_path = os.path.join(
                folder_path,
                filename
            )

            all_pending.append(
                (file_key, h5_path)
            )


# ============================================================
# RESUME PARTIALLY PROCESSED FILE FIRST
# ============================================================

if active_file:

    active_index = next(
        (
            i
            for i, (key, _) in enumerate(all_pending)
            if key == active_file
        ),
        None
    )

    if active_index is None:

        raise RuntimeError(
            f"Active file not found: {active_file}"
        )

    all_pending.insert(
        0,
        all_pending.pop(active_index)
    )


# ============================================================
# RESPECT MAX_FILES AND MAX_TRACES
# ============================================================

remaining_files = max(
    0,
    MAX_FILES - len(processed_files)
)

if count_total >= MAX_TRACES:
    remaining_files = 0

# Invalid files must not consume valid-file slots.
# The actual MAX_FILES limit is checked in the loop.
pending_files = (
    all_pending
    if remaining_files > 0
    else []
)

print()
print(f"Remaining valid-file slots: {remaining_files:,}")
print(f"Pending candidates:          {len(pending_files):,}")
print()


# ============================================================
# PROCESS ALL SAMPLES
# SKIP INVALID HDF5 FILES
# ============================================================

new_files_processed = 0

try:

    for file_key, h5_path in tqdm(
        pending_files,
        desc="TVLA all samples"
    ):

        if count_total >= MAX_TRACES:
            break

        if len(processed_files) >= MAX_FILES:
            break

        # =====================================================
        # CHECK HDF5 SIGNATURE BEFORE OPENING
        # =====================================================

        try:

            valid_hdf5 = h5py.is_hdf5(h5_path)

        except (OSError, ValueError) as error:

            valid_hdf5 = False

            validation_error = str(error)

        else:

            validation_error = "HDF5 signature not recognized"

        if not valid_hdf5:

            # A partially processed file cannot be skipped
            # safely because its data is already in the state.
            if file_key == active_file:

                raise RuntimeError(
                    "The active resume file is invalid: "
                    f"{h5_path}. "
                    "The saved state may already contain "
                    "traces from this file."
                )

            log_invalid_file(
                h5_path,
                validation_error
            )

            continue

        # =====================================================
        # BACKUP ACCUMULATORS FOR SAFE ERROR RECOVERY
        # =====================================================

        backup_sum_f = sum_f.copy()
        backup_sum_r = sum_r.copy()

        backup_sum2_f = sum2_f.copy()
        backup_sum2_r = sum2_r.copy()

        backup_count_f = count_f
        backup_count_r = count_r
        backup_count_total = count_total

        backup_active_file = active_file
        backup_active_offset = active_offset

        backup_new_files = new_files_processed

        try:

            with h5py.File(h5_path, "r") as f:

                # =============================================
                # VERIFY DATASETS EXIST
                # =============================================

                if "traces" not in f:
                    raise ValueError(
                        "Missing dataset: traces"
                    )

                if "labels" not in f:
                    raise ValueError(
                        "Missing dataset: labels"
                    )

                traces_ds = f["traces"]
                labels_ds = f["labels"]

                if not isinstance(traces_ds, h5py.Dataset):
                    raise ValueError(
                        "'traces' is not a dataset"
                    )

                if not isinstance(labels_ds, h5py.Dataset):
                    raise ValueError(
                        "'labels' is not a dataset"
                    )

                # =============================================
                # VERIFY TRACE SHAPE
                # =============================================

                if (
                    traces_ds.ndim != 2
                    or traces_ds.shape[1] != trace_length
                ):

                    raise ValueError(
                        f"Incompatible trace shape: "
                        f"{traces_ds.shape}. "
                        f"Expected: (N, {trace_length})"
                    )

                n_available = traces_ds.shape[0]

                if (
                    labels_ds.ndim != 1
                    or labels_ds.shape[0] != n_available
                ):

                    raise ValueError(
                        f"Traces/labels mismatch: "
                        f"traces={traces_ds.shape}, "
                        f"labels={labels_ds.shape}"
                    )

                if n_available == 0:

                    raise ValueError(
                        "Empty traces dataset"
                    )

                # =============================================
                # RESUME FROM SAVED OFFSET
                # =============================================

                start = (
                    active_offset
                    if file_key == active_file
                    else 0
                )

                if not (0 <= start <= n_available):

                    raise RuntimeError(
                        f"Invalid resume offset: {h5_path}"
                    )

                # =============================================
                # PROCESS IN MEMORY-EFFICIENT BATCHES
                # =============================================

                while (
                    start < n_available
                    and count_total < MAX_TRACES
                ):

                    next_milestone = (
                        count_total // SAVE_EVERY_TRACES + 1
                    ) * SAVE_EVERY_TRACES

                    n_read = min(
                        BATCH_SIZE,
                        n_available - start,
                        MAX_TRACES - count_total,
                        next_milestone - count_total
                    )

                    end = start + n_read

                    # -----------------------------------------
                    # READ ALL SAMPLES
                    # -----------------------------------------

                    traces = np.empty(
                        (n_read, trace_length),
                        dtype=np.float64
                    )

                    traces_ds.read_direct(
                        traces,
                        source_sel=np.s_[start:end, :]
                    )

                    labels = np.asarray(
                        labels_ds[start:end]
                    ).reshape(-1)

                    if len(labels) != n_read:

                        raise ValueError(
                            "Invalid number of labels"
                        )

                    mask_f = labels == 0
                    mask_r = labels == 1

                    if (
                        np.count_nonzero(mask_f)
                        + np.count_nonzero(mask_r)
                        != n_read
                    ):

                        raise ValueError(
                            "Unexpected label values "
                            "(only 0 and 1 are allowed)"
                        )

                    # =========================================
                    # FIXED GROUP
                    # =========================================

                    if np.any(mask_f):

                        tf = traces[mask_f]

                        sum_f += np.sum(
                            tf,
                            axis=0,
                            dtype=np.float64
                        )

                        sum2_f += np.einsum(
                            "ij,ij->j",
                            tf,
                            tf
                        )

                        count_f += tf.shape[0]

                        del tf

                    # =========================================
                    # RANDOM GROUP
                    # =========================================

                    if np.any(mask_r):

                        tr = traces[mask_r]

                        sum_r += np.sum(
                            tr,
                            axis=0,
                            dtype=np.float64
                        )

                        sum2_r += np.einsum(
                            "ij,ij->j",
                            tr,
                            tr
                        )

                        count_r += tr.shape[0]

                        del tr

                    del traces

                    # =========================================
                    # UPDATE PROGRESS
                    # =========================================

                    count_total += n_read
                    start = end

                    active_file = file_key
                    active_offset = start

                    # Mark only fully processed files
                    if start == n_available:

                        processed_files.add(file_key)

                        active_file = ""
                        active_offset = 0

                        new_files_processed += 1

            # =================================================
            # CHECKPOINT AFTER SUCCESSFUL FILE READ
            # =================================================

            # Delaying the checkpoint until the file has
            # been read successfully prevents saving
            # contributions from a subsequently failed file.
            #
            # With 1000 traces/file and milestones of 100000,
            # checkpoints normally coincide with file ends.

            save_checkpoint_if_needed()

        except (OSError, KeyError, ValueError, RuntimeError) as error:

            # =================================================
            # RESTORE ACCUMULATORS AFTER INVALID FILE
            # =================================================

            sum_f[:] = backup_sum_f
            sum_r[:] = backup_sum_r

            sum2_f[:] = backup_sum2_f
            sum2_r[:] = backup_sum2_r

            count_f = backup_count_f
            count_r = backup_count_r
            count_total = backup_count_total

            active_file = backup_active_file
            active_offset = backup_active_offset

            new_files_processed = backup_new_files

            processed_files.discard(file_key)

            # Do not silently discard a file that was
            # partially counted in a previous saved state.
            if file_key == backup_active_file:

                raise RuntimeError(
                    "The resumed active file failed. "
                    "Its previous contributions cannot be "
                    "safely discarded from the saved state. "
                    f"File: {h5_path}"
                ) from error

            log_invalid_file(
                h5_path,
                f"{type(error).__name__}: {error}"
            )

            continue

        finally:

            del backup_sum_f
            del backup_sum_r
            del backup_sum2_f
            del backup_sum2_r

        if count_total >= MAX_TRACES:
            break

finally:

    # Always save latest state
    save_state(STATE_FILE)

    print()
    print("Latest TVLA state saved:")
    print(os.path.abspath(STATE_FILE))


# ============================================================
# EXECUTION SUMMARY
# ============================================================

print()
print("=" * 60)
print("EXECUTION SUMMARY")
print("=" * 60)

print(f"New files processed:    {new_files_processed:,}")
print(f"New traces processed:   {count_total - initial_count_total:,}")

print()
print(f"Total valid files:      {len(processed_files):,}")
print(f"Total traces:           {count_total:,}")

print(f"Fixed traces:           {count_f:,}")
print(f"Random traces:          {count_r:,}")

print(f"Active file:            {active_file or 'None'}")
print(f"Active offset:          {active_offset:,}")

print()
print(f"Invalid files skipped:  {len(skipped_invalid_files):,}")

if skipped_invalid_files:

    print("\nSKIPPED FILES:")
    print("-" * 60)

    for entry in skipped_invalid_files:
        print(entry)

    print()
    print("Skip log:")
    print(os.path.abspath(SKIPPED_LOG_FILE))

print("=" * 60)


# ============================================================
# FINAL TVLA
# ============================================================

t_values = calculate_t_values()

valid = np.isfinite(t_values)

n_leaking = np.count_nonzero(
    valid & (np.abs(t_values) > TVLA_THRESHOLD)
)

print()
print("=" * 60)
print("TVLA RESULTS - ALL SAMPLES")
print("=" * 60)

print(f"Total traces: {count_total:,}")
print(f"Fixed traces: {count_f:,}")
print(f"Random traces: {count_r:,}")

if np.any(valid):

    max_index = int(
        np.nanargmax(np.abs(t_values))
    )

    max_t = abs(t_values[max_index])

    print(f"Max |t|: {max_t:.6f}")
    print(f"Sample with max |t|: {max_index:,}")

else:

    print(
        "TVLA undefined: insufficient data "
        "or zero variance."
    )

print(
    f"Samples with |t| > {TVLA_THRESHOLD}: "
    f"{n_leaking:,} / {trace_length:,}"
)


# ============================================================
# PLOT ALL SAMPLES
# ============================================================

plt.figure(figsize=(16, 5))

plt.plot(
    t_values,
    linewidth=0.8
)

plt.axhline(
    TVLA_THRESHOLD,
    color="red",
    linestyle="--"
)

plt.axhline(
    -TVLA_THRESHOLD,
    color="red",
    linestyle="--"
)

plt.xlabel("Sample")
plt.ylabel("Welch t-value")

plt.title(
    "First-order TVLA - Fixed vs Random - All Samples"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
